# Practice: probability, simulation and a real spam filter
*AI from scratch, probability 1* · lesson: https://ihor.sh/ai/p1-probability-and-bayes.html

Answer probability questions by simulation (a million trials take a fraction of a second in NumPy), check them
against the exact formulas, apply Bayes' rule, and train a naive Bayes spam filter on 5,574 real text messages.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).
On Kaggle, turn on *Internet* in the notebook settings for part 5's download.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['** is "to the power of": (5 / 6) ** 4.', 'p_exact = 1 - (5 / 6) ** 4'], 2: ['(rolls == 6).any(axis=1) is one True/False per roll; the mean of True/False is the share that are True.', 'p_sim = (rolls == 6).any(axis=1).mean()'], 3: ['Sample random birthdays (100_000 rooms, 23 people each) from 0 to 364.', 'Sort each room along axis 1 using np.sort(..., axis=1).', 'Check for equal adjacent days with (np.diff(days, axis=1) == 0).any(axis=1).', 'Take the mean across all rooms and assign to p_shared.'], 4: ['Compute p_stay = (pick == car).mean() (win if initial pick was already the car).', 'Compute p_switch = (pick != car).mean() (win if initial pick was a goat, because host reveals the other).'], 5: ['Compute P(+ and condition) = sensitivity * prior.', 'Compute P(+ and healthy) = false_positive * (1 - prior).', 'Return P(+ and condition) / (P(+ and condition) + P(+ and healthy)).'], 6: ['Split sms.text and (sms.label == "spam") using train_test_split(..., test_size=0.2, random_state=0).', 'Initialize CountVectorizer(), fit_transform on Xtr, transform on Xte.', 'Fit MultinomialNB().fit(Xtr_vec, ytr) and predict labels on Xte_vec.', 'Compute test accuracy = (pred == yte).mean() and spam_recall = pred[yte.values].mean().'], 7: ['math.comb(4, 2) counts the orders with 2 heads (6); each order has chance 0.5 ** 4.', 'rng.binomial(4, 0.5, 1_000_000) gives the heads count of each round; == 2 then .mean() is the share with exactly 2.']}
_ANSWERS = {1: 'cF9leGFjdCA9IDEgLSAoNSAvIDYpICoqIDQ=', 2: 'cF9zaW0gPSAocm9sbHMgPT0gNikuYW55KGF4aXM9MSkubWVhbigp', 3: 'ZGF5cyA9IG5wLnNvcnQocm5nLmludGVnZXJzKDAsIDM2NSwgc2l6ZT0oMTAwXzAwMCwgMjMpKSwgYXhpcz0xKQpwX3NoYXJlZCA9IChucC5kaWZmKGRheXMsIGF4aXM9MSkgPT0gMCkuYW55KGF4aXM9MSkubWVhbigp', 4: 'cF9zdGF5ID0gKHBpY2sgPT0gY2FyKS5tZWFuKCkKcF9zd2l0Y2ggPSAocGljayAhPSBjYXIpLm1lYW4oKQ==', 5: 'cmV0dXJuIHNlbnNpdGl2aXR5ICogcHJpb3IgLyAoc2Vuc2l0aXZpdHkgKiBwcmlvciArIGZhbHNlX3Bvc2l0aXZlICogKDEgLSBwcmlvcikp', 6: 'WHRyLCBYdGUsIHl0ciwgeXRlID0gdHJhaW5fdGVzdF9zcGxpdChzbXMudGV4dCwgc21zLmxhYmVsID09ICJzcGFtIiwgdGVzdF9zaXplPTAuMiwgcmFuZG9tX3N0YXRlPTApCnZlYyA9IENvdW50VmVjdG9yaXplcigpCm1vZGVsID0gTXVsdGlub21pYWxOQigpLmZpdCh2ZWMuZml0X3RyYW5zZm9ybShYdHIpLCB5dHIpCnByZWQgPSBtb2RlbC5wcmVkaWN0KHZlYy50cmFuc2Zvcm0oWHRlKSkKYWNjdXJhY3kgPSAocHJlZCA9PSB5dGUpLm1lYW4oKQpzcGFtX3JlY2FsbCA9IHByZWRbeXRlLnZhbHVlc10ubWVhbigp', 7: 'cDIgPSBtYXRoLmNvbWIoNCwgMikgKiAwLjUgKiogNApwMl9zaW0gPSAocm5nLmJpbm9taWFsKDQsIDAuNSwgMV8wMDBfMDAwKSA9PSAyKS5tZWFuKCk='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import math
import numpy as np

rng = np.random.default_rng(0)

## 1. At least one six
Roll 4 dice. The chance of no six on one die is 5/6, so of no six on all four, $(5/6)^4$, and of at least one six
the complement, $1 - (5/6)^4$. Set `p_exact` to that.

In [ ]:
p_exact = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print(p_exact)

In [ ]:
check(1, lambda: abs(p_exact - 0.5177) < 1e-3, "1 − (5/6)⁴ is about 0.5177")

Now check it by simulation: a million rolls of 4 dice, one roll per row. `rolls == 6` marks every six True;
`.any(axis=1)` asks of each row "is any of them True?". Set `p_sim` to the fraction of rows with at least one six.

In [ ]:
rolls = rng.integers(1, 7, size=(1_000_000, 4))
print(rolls[:3])
p_sim = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print(p_sim, p_exact)

In [ ]:
check(2, lambda: abs(p_sim - p_exact) < 0.003, "a million rolls should land within 0.003 of the exact 0.5177")

## 2. The birthday problem, simulated
Fill 100,000 rooms of 23 people with random birthdays (0–364). Set `p_shared` to the fraction of rooms where two
people share one. Hint: sort each row (`np.sort(..., axis=1)`), then look for equal neighbors with `diff` (`np.diff(...) == 0`).

In [ ]:
p_shared = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
print(p_shared)

In [ ]:
check(3, lambda: abs(p_shared - 0.507) < 0.01, 
      "With 23 people, the probability of at least one shared birthday should be ~50.7%.")

## 3. Monty Hall, a million games at once
`car` and `pick` are random doors (0–2) for each game. When you switch, you win exactly when your first pick was
wrong (the host opens the other goat door). Set `p_stay` and `p_switch` to the win rates.

In [ ]:
n = 1_000_000
car, pick = rng.integers(0, 3, n), rng.integers(0, 3, n)
p_stay = p_switch = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print(p_stay, p_switch)

In [ ]:
check(4, lambda: abs(p_stay - 1 / 3) < 0.003 and abs(p_switch - 2 / 3) < 0.003, 
      "Staying should yield ~33.3% wins (1/3) and switching should yield ~66.7% wins (2/3).")

## 4. Bayes' rule
Write `posterior(prior, sensitivity, false_positive)`: the probability of having the condition after a positive
test, $\frac{P(+\mid\text{sick})P(\text{sick})}{P(+\mid\text{sick})P(\text{sick}) + P(+\mid\text{healthy})P(\text{healthy})}$.

In [ ]:
def posterior(prior, sensitivity, false_positive):
    """Compute P(condition | positive test) via Bayes' theorem.
    
    Args:
        prior: Float P(condition), the base rate of the condition in the population.
        sensitivity: Float P(+ | condition), true positive rate.
        false_positive: Float P(+ | healthy), false positive rate.
    Returns:
        Float posterior probability P(condition | +).
    """
    # YOUR CODE HERE (stuck? hint(5), then answer(5))
    ...

print(posterior(0.01, 0.9, 0.09))

In [ ]:
check(5, lambda: abs(posterior(0.01, 0.9, 0.09) - 0.0917) < 1e-3, 
      "For a rare condition (1% prior, 90% sensitivity, 9% false positive), posterior should be ~9.17%.")
check(5, lambda: posterior(0.01, 0.9, 0.01) > 0.45, 
      "Reducing the false positive rate to 1% should dramatically raise the posterior above 45%.")

## 5. A real spam filter
The SMS Spam Collection (Almeida & Hidalgo, UCI, CC BY 4.0): 5,574 real text messages labeled spam or ham.

In [ ]:
import io, urllib.request, zipfile
import pandas as pd
raw = urllib.request.urlopen("https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip").read()
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    sms = pd.read_csv(z.open("SMSSpamCollection"), sep="\t", names=["label", "text"], quoting=3)
print(sms.label.value_counts()); sms.head()

Train naive Bayes on 80% and test on the held-out 20%:
1. split messages with `.text` via `train_test_split(sms.text, sms.label == "spam", test_size=0.2, random_state=0)`
2. turn texts into word counts with `CountVectorizer` (`fit_transform` on train, only `transform` on test)
3. fit `MultinomialNB()` and set `accuracy` on the test set, plus `spam_recall`: the share of real spam it caught.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
accuracy = spam_recall = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
print("accuracy", accuracy, " spam caught", spam_recall)

In [ ]:
check(6, lambda: accuracy > 0.97, 
      "Naive Bayes test accuracy should exceed 97% on SMS spam classification.")
check(6, lambda: spam_recall > 0.85, 
      "Spam recall (fraction of actual spam correctly caught) should exceed 85%.")

Why is "accuracy" not enough here? Only about 13% of messages are spam: a filter that never flags anything
scores 87%. Chapter 12 (measuring a model) is about exactly this. Which words push hardest toward spam?

In [ ]:
if accuracy is not None:
    log_ratio = model.feature_log_prob_[1] - model.feature_log_prob_[0]
    words = np.array(vec.get_feature_names_out())
    print("spammiest words:", list(words[np.argsort(log_ratio)[-15:]][::-1]))

## 6. The binomial
Set `p2` to the exact chance of exactly 2 heads in 4 fair flips with `comb` (`math.comb`), and `p2_sim` by simulating
1,000,000 rounds with `rng.binomial(4, 0.5, 1_000_000)`.

In [ ]:
p2 = p2_sim = None
# YOUR CODE HERE (stuck? hint(7), then answer(7))
...
print(p2, p2_sim)

In [ ]:
check(7, lambda: p2 == 0.375 and abs(p2_sim - 0.375) < 0.003, "6 orders × (1/2)⁴ = 0.375")

## Next
Back to the path: https://ihor.sh/ai/ · next: probability 2, describing data.